In [5]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.interpolate import interp1d

In [2]:
import pandas as pd
import numpy as np
import os

def load_wesad_data(directory):
    data = {}
    for participant_id in os.listdir(directory):
        participant_path = os.path.join(directory, participant_id)
        if os.path.isdir(participant_path):
            participant_data = {}
            for file_name in os.listdir(participant_path):
                file_path = os.path.join(participant_path, file_name)
                if file_name.endswith('.pkl'):
                    participant_data[file_name.split('.')[0]] = pd.read_pickle(file_path)
            data[participant_id] = participant_data
    return data

wesad_data = load_wesad_data('C:/Users/rusha/Desktop/Uni_Freiburg_Notes/MDD/WESAD')

In [18]:
import numpy as np
from scipy.interpolate import interp1d
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def interpolate_signal(signal, original_rate, target_rate, target_length):
    """
    Interpolates the signal to match the target length.
    
    Parameters:
    - signal: The original signal array.
    - original_rate: Sampling rate of the original signal.
    - target_rate: Desired sampling rate (corresponds to label frequency).
    - target_length: Length of the interpolated signal (number of target samples).
    
    Returns:
    - Interpolated signal.
    """
    original_length = len(signal)
    
    # generate time vectors
    time_original = np.linspace(0, original_length / original_rate, original_length)
    time_target = np.linspace(0, original_length / original_rate, target_length)
    
    # create an interpolation function
    interpolator = interp1d(time_original, signal, kind='linear', fill_value='extrapolate')
    
    # interpolate to the new length
    interpolated_signal = interpolator(time_target)
    
    return interpolated_signal

def extract_and_process_data(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    wrist_data = subject_details['signal']['wrist']
    labels = subject_details['label']
    
    # include only valid ones
    valid_labels = np.isin(labels, [0, 1, 2, 3, 4])
    labels = labels[valid_labels]
    
    num_samples = len(labels)  # Number of samples in the labels
    
    # Interpolate each data component to match the length of labels
    bvp_interpolated = interpolate_signal(wrist_data['BVP'].flatten(), sampling_rates['BVP'], sampling_rates['label'], num_samples)
    eda_interpolated = interpolate_signal(wrist_data['EDA'].flatten(), sampling_rates['EDA'], sampling_rates['label'], num_samples)
    temp_interpolated = interpolate_signal(wrist_data['TEMP'].flatten(), sampling_rates['TEMP'], sampling_rates['label'], num_samples)
    
    # reshape each feature to ensure proper concatenation
    bvp_interpolated = bvp_interpolated.reshape(-1, 1)
    eda_interpolated = eda_interpolated.reshape(-1, 1)
    temp_interpolated = temp_interpolated.reshape(-1, 1)
    
    print(f"BVP shape: {bvp_interpolated.shape}")
    print(f"EDA shape: {eda_interpolated.shape}")
    print(f"TEMP shape: {temp_interpolated.shape}")
    
    # combine the interpolated features into a single array
    features = np.hstack((
        bvp_interpolated,
        eda_interpolated,
        temp_interpolated
    ))

    return features, labels

# sampling rates
sampling_rates = {
    'BVP': 64,
    'EDA': 4,
    'TEMP': 4,
    'label': 700
}

# extract and process data from subjects
features_s10, labels_s10 = extract_and_process_data('S10')
features_s11, labels_s11 = extract_and_process_data('S11')

# combine features and labels from both subjects
combined_features = np.vstack((features_s10, features_s11))
combined_labels = np.hstack((labels_s10, labels_s11))

# check the initial label distribution
initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

# remove labels 5, 6, 7
valid_labels = np.isin(combined_labels, [0, 1, 2, 3, 4])
balanced_features = combined_features[valid_labels]
balanced_labels = combined_labels[valid_labels]

# determine the minimum number of samples for any class
min_samples_per_class = min(Counter(balanced_labels).values())

# create balanced dataset by randomly sampling min_samples_per_class instances from each class
balanced_features_list = []
balanced_labels_list = []

for label in np.unique(balanced_labels):
    label_indices = np.where(balanced_labels == label)[0]
    if len(label_indices) > min_samples_per_class:
        sampled_indices = np.random.choice(label_indices, min_samples_per_class, replace=False)
    else:
        sampled_indices = label_indices
    balanced_features_list.append(balanced_features[sampled_indices])
    balanced_labels_list.append(balanced_labels[sampled_indices])

# convert the balanced features and labels to numpy arrays
balanced_features = np.vstack(balanced_features_list)
balanced_labels = np.hstack(balanced_labels_list)

# check the label distribution after balancing
balanced_label_distribution = Counter(balanced_labels)
print("Balanced label distribution:", balanced_label_distribution)


X_train, X_test, y_train, y_test = train_test_split(balanced_features, balanced_labels, test_size=0.2, random_state=42, stratify=balanced_labels)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# check the label distribution in training set
label_distribution_train = Counter(y_train)
print("Label distribution in the training set:", label_distribution_train)

# check the label distribution in test set
label_distribution_test = Counter(y_test)
print("Label distribution in the test set:", label_distribution_test)


BVP shape: (3740100, 1)
EDA shape: (3740100, 1)
TEMP shape: (3740100, 1)
BVP shape: (3556701, 1)
EDA shape: (3556701, 1)
TEMP shape: (3556701, 1)
Initial label distribution: Counter({0: 3032400, 1: 1652000, 4: 1110901, 2: 983500, 3: 518000})
Balanced label distribution: Counter({0: 518000, 1: 518000, 2: 518000, 3: 518000, 4: 518000})
Accuracy: 0.9895849420849421
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98    103600
           1       0.99      0.99      0.99    103600
           2       1.00      1.00      1.00    103600
           3       0.98      0.99      0.98    103600
           4       0.99      1.00      0.99    103600

    accuracy                           0.99    518000
   macro avg       0.99      0.99      0.99    518000
weighted avg       0.99      0.99      0.99    518000

Label distribution in the training set: Counter({1: 414400, 4: 414400, 2: 414400, 0: 414400, 3: 414400})
Label distribution

In [8]:
# extract and process data from subjects S10 and S11
features_s10, labels_s10 = extract_and_process_data('S10')
features_s11, labels_s11 = extract_and_process_data('S11')

# combine features and labels from both subjects
combined_features = np.vstack((features_s10, features_s11))
combined_labels = np.hstack((labels_s10, labels_s11))

# check the initial label distribution
initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

# remove labels 5, 6, 7 since thy're not relevant
valid_labels = np.isin(combined_labels, [0, 1, 2, 3, 4])
balanced_features = combined_features[valid_labels]
balanced_labels = combined_labels[valid_labels]

# determine the minimum number of samples for any class
min_samples_per_class = min(Counter(balanced_labels).values())

# create balanced dataset by randomly sampling min_samples_per_class instances from each class
balanced_features_list = []
balanced_labels_list = []

for label in np.unique(balanced_labels):
    label_indices = np.where(balanced_labels == label)[0]
    sampled_indices = np.random.choice(label_indices, min_samples_per_class, replace=False)
    balanced_features_list.append(balanced_features[sampled_indices])
    balanced_labels_list.append(balanced_labels[sampled_indices])

# convert the balanced features and labels to numpy arrays
balanced_features = np.vstack(balanced_features_list)
balanced_labels = np.hstack(balanced_labels_list)

# check the label distribution after balancing
balanced_label_distribution = Counter(balanced_labels)
print("Balanced label distribution:", balanced_label_distribution)

X_train, X_test, y_train, y_test = train_test_split(balanced_features, balanced_labels, test_size=0.2, random_state=42, stratify=balanced_labels)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)


y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# check the label distribution in the training set
label_distribution_train = Counter(y_train)
print("Label distribution in the training set:", label_distribution_train)

# check the label distribution in the test set
label_distribution_test = Counter(y_test)
print("Label distribution in the test set:", label_distribution_test)

ValueError: x and y arrays must be equal in length along interpolation axis.

In [ ]:
# function for extracting and combining wrist data and labels for a given subject
def extract_and_process_data(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    wrist_data = subject_details['signal']['wrist']
    labels = subject_details['label']

    # ensure labels are within valid range
    valid_labels = np.isin(labels, [0, 1, 2, 3, 4])
    labels = labels[valid_labels]
    
    # downsample the features to match the labels length
    def downsample(signal, target_length):
        return signal[:target_length]  

    num_samples = len(labels)  # number of samples in the labels

    features = np.hstack((
        downsample(wrist_data['ACC'], num_samples),
        downsample(wrist_data['BVP'], num_samples),
        downsample(wrist_data['EDA'], num_samples),
        downsample(wrist_data['TEMP'], num_samples)
    ))

    return features, labels

# extract and process data from subjects S10 and S11
features_s10, labels_s10 = extract_and_process_data('S10')
features_s11, labels_s11 = extract_and_process_data('S11')

# combine features and labels from both subjects
combined_features = np.vstack((features_s10, features_s11))
combined_labels = np.hstack((labels_s10, labels_s11))

initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

valid_labels = np.isin(combined_labels, [0, 1, 2, 3, 4])
balanced_features = combined_features[valid_labels]
balanced_labels = combined_labels[valid_labels]

min_samples_per_class = min(Counter(balanced_labels).values())

balanced_features_list = []
balanced_labels_list = []

In [ ]:
for label in np.unique(balanced_labels):
    label_indices = np.where(balanced_labels == label)[0]
    sampled_indices = np.random.choice(label_indices, min_samples_per_class, replace=False)
    balanced_features_list.append(balanced_features[sampled_indices])
    balanced_labels_list.append(balanced_labels[sampled_indices])

balanced_features = np.vstack(balanced_features_list)
balanced_labels = np.hstack(balanced_labels_list)

In [ ]:

balanced_label_distribution = Counter(balanced_labels)
print("Balanced label distribution:", balanced_label_distribution)

X_train, X_test, y_train, y_test = train_test_split(balanced_features, balanced_labels, test_size=0.2, random_state=42, stratify=balanced_labels)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:

clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

label_distribution_train = Counter(y_train)
print("Label distribution in the training set:", label_distribution_train)

label_distribution_test = Counter(y_test)
print("Label distribution in the test set:", label_distribution_test)